In [30]:
!pip install nltk

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re
from collections import Counter #Counts frequency of elements in a list or text.
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize,sent_tokenize  #Splits text into individual words/tokens.
import nltk # Used for NLP tasks like tokenization, stemming, stopword removal, etc

In [32]:
# NLTK does NOT come with pretrained tokenizers by default.

# Tokenization
nltk.download('punkt')

# Some newer NLTK versions require additional tokenizer tables.
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [34]:
from sklearn.datasets import fetch_20newsgroups
data = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
docs = data.data[:100]

## Tokenized

In [36]:
def tokenize(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 3. Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # 4. Remove quotes and apostrophes
    text = re.sub(r'["\'`]', '', text)

    # 5. Remove punctuation and special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 6. Remove extra spaces, tabs, and new lines
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Split into words
    tokens = text.split()

    return tokens

In [37]:
clean_docs = [tokenize(doc) for doc in docs]
print(clean_docs[0])

['i', 'was', 'wondering', 'if', 'anyone', 'out', 'there', 'could', 'enlighten', 'me', 'on', 'this', 'car', 'i', 'saw', 'the', 'other', 'day', 'it', 'was', 'a', '2', 'door', 'sports', 'car', 'looked', 'to', 'be', 'from', 'the', 'late', '60s', 'early', '70s', 'it', 'was', 'called', 'a', 'bricklin', 'the', 'doors', 'were', 'really', 'small', 'in', 'addition', 'the', 'front', 'bumper', 'was', 'separate', 'from', 'the', 'rest', 'of', 'the', 'body', 'this', 'is', 'all', 'i', 'know', 'if', 'anyone', 'can', 'tellme', 'a', 'model', 'name', 'engine', 'specs', 'years', 'of', 'production', 'where', 'this', 'car', 'is', 'made', 'history', 'or', 'whatever', 'info', 'you', 'have', 'on', 'this', 'funky', 'looking', 'car', 'please', 'e', 'mail']


# Build Vocabs

In [38]:

vocab = { '<pad>': 0,'<unk>': 1}
for doc in clean_docs:
    for token in doc:
        if token not in vocab:
            vocab[token] = len(vocab)


In [39]:
len(vocab)

4932

# Text to Number

In [40]:
# 1. Combine all documents into one string
all_text = '\n'.join(docs)
# 2. Normalize whitespace
all_text = re.sub(r'\s+', ' ', all_text).strip()
# 3. Split into sentences
input_sentences = sent_tokenize(all_text)
# 4. Remove empty sentences
input_sentences = [
    sentence.strip()
    for sentence in input_sentences
    if sentence.strip()
]

In [41]:
len(input_sentences)

1098

In [42]:
def text_to_indices(sentence,vocab):
  numerical_sentence=[]
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])
  return numerical_sentence


In [43]:
input_numerical_sentence= []
for sentence in input_sentences:
  input_numerical_sentence.append(text_to_indices(tokenize(sentence),vocab))


In [44]:
len(input_numerical_sentence)

1098

# training Sequence form

In [45]:
training_sequence = []
for sentence in input_numerical_sentence:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])

In [46]:
len(training_sequence)

18966

In [47]:
max_len = max(len(sequence) for sequence in training_sequence)
print(max_len)

135


# Padding

In [48]:
padded_traning_squence=[]
for sequence in training_sequence:
  padded_traning_squence.append([vocab['<pad>']]*(max_len-len(sequence))+sequence)

In [49]:
len(padded_traning_squence[0])

135

# Tensor and Split

In [50]:
padded_traning_squence=torch.tensor(padded_traning_squence,dtype=torch.long)

In [51]:
X =padded_traning_squence[:, :-1]

In [52]:
y = padded_traning_squence[:,-1]

#Custom Dataset and Dataloader

In [53]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx) :
    return self.X[idx], self.y[idx]

In [54]:
dataset = CustomDataset(X,y)

In [55]:
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

# LSTM Cell

In [56]:
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,100)
    self.lstm = nn.LSTM(100,150,batch_first=True)
    self.fc = nn.Linear(150,vocab_size)

  def forward(self,x):
    embedded = self.embedding(x)
    intermediate_hidden_states,(final_hidden_state,final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output


In [57]:
model = LSTMModel(len(vocab))
model.to(device)

LSTMModel(
  (embedding): Embedding(4932, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=4932, bias=True)
)

# Traning Loop

In [58]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# training loop

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 4388.7492
Epoch: 2, Loss: 3901.5626
Epoch: 3, Loss: 3452.1121
Epoch: 4, Loss: 2958.9337
Epoch: 5, Loss: 2491.6797
Epoch: 6, Loss: 2068.1280
Epoch: 7, Loss: 1697.2971
Epoch: 8, Loss: 1384.6156
Epoch: 9, Loss: 1130.8767
Epoch: 10, Loss: 920.1896
Epoch: 11, Loss: 747.8538
Epoch: 12, Loss: 607.0848
Epoch: 13, Loss: 491.8162
Epoch: 14, Loss: 399.5094
Epoch: 15, Loss: 326.2388
Epoch: 16, Loss: 268.4454
Epoch: 17, Loss: 226.0484
Epoch: 18, Loss: 193.1177
Epoch: 19, Loss: 169.1137
Epoch: 20, Loss: 152.9481
Epoch: 21, Loss: 139.4620
Epoch: 22, Loss: 131.5378
Epoch: 23, Loss: 124.0560
Epoch: 24, Loss: 118.8216
Epoch: 25, Loss: 115.5992
Epoch: 26, Loss: 112.0557
Epoch: 27, Loss: 110.7729
Epoch: 28, Loss: 107.7567
Epoch: 29, Loss: 104.2720
Epoch: 30, Loss: 103.9901
Epoch: 31, Loss: 101.2355
Epoch: 32, Loss: 101.9751
Epoch: 33, Loss: 108.0203
Epoch: 34, Loss: 101.5016
Epoch: 35, Loss: 97.2844
Epoch: 36, Loss: 95.4712
Epoch: 37, Loss: 94.9682
Epoch: 38, Loss: 93.7875
Epoch: 39, Loss:

## Prediction

In [66]:
def prediction(model,vocab,text):
  model.eval()

  #tokenize
  tokenized_text = tokenize(text)

  #numerical indices
  numerical_text = text_to_indices(tokenized_text,vocab)

  #padding
  padded_text = torch.tensor([vocab['<pad>']]* (max_len-len(numerical_text)) + numerical_text , dtype=torch.long).unsqueeze(0).to(device)

  #send to model
  output = model(padded_text)

  #predicted index

  value, index = torch.max(output,dim=1)

  #merge with text

  return text + " " + list(vocab.keys())[index]





In [69]:
prediction(model,vocab," If anyone can tellme a")

' If anyone can tellme a model'

# Accuracy calculation

In [70]:
def calculate_accuracy(model,dataloader,device):

  model.eval() #set the model to evaluation mode

  correct = 0
  total = 0

  with torch.no_grad():
    for batch_x, batch_y in dataloader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      #get model prediction
      output = model(batch_x)

      #get the predictec word indices

      _, predicted = torch.max(output,dim=1)

      #compare wityh actual label

      correct += (predicted == batch_y).sum().item()

      total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy






In [71]:
accuracy = calculate_accuracy(model,dataloader,device)

print(f" Model accuracy {accuracy :.2f}%")

 Model accuracy 95.85%


# Generate Automatically

In [73]:
import time
num_tokens = 20
input_text= 'It was a 2-door sports car,'

for i in range(num_tokens):
  output_text = prediction(model,vocab,input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)

It was a 2-door sports car, looked
It was a 2-door sports car, looked to
It was a 2-door sports car, looked to be
It was a 2-door sports car, looked to be from
It was a 2-door sports car, looked to be from the
It was a 2-door sports car, looked to be from the late
It was a 2-door sports car, looked to be from the late 60s
It was a 2-door sports car, looked to be from the late 60s early
It was a 2-door sports car, looked to be from the late 60s early 70s
It was a 2-door sports car, looked to be from the late 60s early 70s 70s
It was a 2-door sports car, looked to be from the late 60s early 70s 70s lines
It was a 2-door sports car, looked to be from the late 60s early 70s 70s lines and
It was a 2-door sports car, looked to be from the late 60s early 70s 70s lines and then
It was a 2-door sports car, looked to be from the late 60s early 70s 70s lines and then used
It was a 2-door sports car, looked to be from the late 60s early 70s 70s lines and then used muslim
It was a 2-door sports car